# 02 - 9 级策略链详解

> **何时使用**: 当你需要自定义每列的数据类型，或者对自动推断的结果不满意时。
>
> **核心概念**: sqlseed 的 `ColumnMapper` 按 9 级优先级自动匹配列名 → 生成器。理解这个链条，你就能精确控制每列的数据。

## 适用场景

- 列名不是标准命名（如 `user_name` 而非 `name`），需要手动指定生成器
- 需要限制数据范围（如 `age` 在 18-65 之间）
- 需要生成特定模式的数据（如订单号 `ORD-\d{6}`）
- 想了解 sqlseed 的自动推断逻辑

## 你将学到

- 9 级策略链的完整优先级
- 每级的匹配逻辑和触发条件
- 74 条精确匹配规则 + 26 条正则模式
- 如何用 `columns={}` 覆盖自动推断

详见 architecture.zh-CN.md §3

**📚 教程导航**

| 序号 | 主题 | 架构层 | 前置要求 |
|------|------|--------|----------|
| 01 | 快速上手与核心流程 | Orchestrator | 无 |
| **→ 02** | **9 级策略链详解** | **Core: ColumnMapper** | **01** |
| 03 | 生成器与 Provider 体系 | Generators | 01 |
| 04 | 数据库层与多表关联 | Database + Core | 01 |
| 05 | 表达式派生与约束求解 | Core: DAG / Expression | 01 |
| 06 | 配置驱动与 Transform | Config / Core | 01 |
| 07 | AI 智能配置 | Plugins: AI | 01 |
| 08 | MCP 服务器集成 | Plugins: MCP | 07 |
| 09 | 插件系统与 Hook 生命周期 | Plugins | 01 |
| 10 | CLI 参考手册 | CLI | 06 |
| 11 | 工具类参考 | Utils | 01 |
| 12 | 测试集成模式 | Testing | 01 |

---

In [1]:
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys; sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 架构定位

| 模块 | 文件 | 核心类/函数 |
|------|------|------------|
| Schema 推断 | `src/sqlseed/core/schema.py` | `SchemaInferrer` |
| 列映射 | `src/sqlseed/core/mapper.py` | `ColumnMapper.map_column()` |

> 对应架构图: [§3 ColumnMapper 9 级策略链](../docs/architecture.zh-CN.md#3-columnmapper-9-级策略链)

## 1. 先看效果 — 零配置的魔法

sqlseed 最强大的特性：**无需任何配置**，它能根据列名自动推断出正确的数据类型。列名叫 `email`？生成邮箱。列名叫 `name`？生成姓名。列名叫 `created_at`？生成时间戳。

先看效果，再解释原理：

In [2]:
# 零配置！sqlseed 根据列名自动选择生成器
rows = preview(str(db_path), table="members", count=3)
print(f"{'name':<18s}  {'email':<28s}  {'phone':<18s}  {'org_code':<10s}")
print('-' * 78)
for row in rows:
    print(f"{row.get('name', 'N/A'):<18s}  {row.get('email', 'N/A'):<28s}  {row.get('phone', 'N/A'):<18s}  {row.get('org_code', 'N/A'):<10s}")  # noqa: E501

name                email                         phone               org_code  
------------------------------------------------------------------------------
Scott Mcmahon       committed2091@example.org     +1-386-559-5630     fLBcbfnoGM
Claud Reese         settings1825@example.com      +17728572576        JmTPSI    
Neil Mercer         ridge2025@duck.com            +18653406653        fLBcbfnoGM


没有写任何映射配置 — sqlseed 的 `ColumnMapper` 自动完成了：

- `name` 列 → 匹配 `name` 规则 → 生成真实姓名
- `email` 列 → 匹配 `email` 规则 → 生成邮箱地址
- `phone` 列 → 匹配 `phone` 规则 → 生成电话号码
- `member_no` 列 → 匹配 UNIQUE 约束 → 生成唯一编号

背后的秘密是 **9 级策略链** — sqlseed 按优先级逐级尝试，直到找到匹配的生成器。

## 2. 策略链概览

sqlseed 的 `ColumnMapper` 按以下优先级依次尝试匹配列名：

| 级别 | 策略 | 说明 |
|:----:|------|------|
| 1 | Autoincrement PK | 自增主键自动跳过 |
| 2 | User Config | 用户显式配置覆盖一切 |
| 3 | Custom Exact Match | 插件注册的精确规则 |
| 4 | Built-in Exact Match | 74 条内置精确匹配规则 |
| 5 | DEFAULT Value | 有默认值的列跳过或 enrich |
| 6 | Custom Pattern Match | 插件注册的正则规则 |
| 7 | Built-in Pattern Match | 26 条内置正则模式匹配 |
| 8 | Nullable | 可空列跳过或 enrich |
| 9 | Type Fallback | 22 种 SQL 类型忠实回退 |

一旦某级匹配成功，后续级别不再执行。Level 3 和 6 是插件扩展点，详见 08-plugin-hooks.ipynb。

## 3. Level 1: Autoincrement PK

如果列是主键且为自增（或类型为 INTEGER/INT），自动跳过，不生成数据。

In [3]:
from sqlseed import preview

rows = preview(str(db_path), table="members", count=2)
for row in rows:
    print(f"name={row['name']}, email={row['email']}")
print("\nmember_id 是自增主键,preview 不包含该列（由 SQLite 自动分配）")

name=Lien Duke, email=carb1837@protonmail.com
name=Willodean Hoover, email=cube2042@duck.com

member_id 是自增主键,preview 不包含该列（由 SQLite 自动分配）


## 4. Level 2: User Config

用户通过 `columns` 参数或 YAML 配置显式指定生成器，优先级最高（仅次于自增主键跳过）。

In [4]:
rows = preview(
    str(db_path),
    table="members",
    count=2,
    columns={
        "name": {"generator": "pattern", "params": {"regex": "User-\\d{4}"}},
        "balance": {"generator": "float", "params": {"min_value": 1000.0, "max_value": 5000.0}},
    },
)
for row in rows:
    print(f"name={row['name']}, balance={row['balance']}")
print("\nname 和 balance 被用户配置覆盖,不再走自动推断")

name=User-3435, balance=4686.8
name=User-7312, balance=3861.57

name 和 balance 被用户配置覆盖,不再走自动推断


## 5. Level 4: Built-in Exact Match（74 条规则）

精确匹配是最常用的策略。sqlseed 内置了 74 条列名到生成器的映射规则。

### 语义类（推断业务含义）

| 列名 | 生成器 | 参数 |
|------|--------|------|
| `email` | email | - |
| `phone` | phone | - |
| `name` | name | - |
| `address` | address | - |
| `city` | city | - |
| `country` | country | - |
| `url` / `website` | url | - |
| `password` | password | - |
| `uuid` | uuid | - |

### 数值类（带合理范围）

| 列名 | 生成器 | 参数 |
|------|--------|------|
| `age` | integer | 18-65 |
| `balance` | float | 0-999999.99 |
| `salary` | float | 3000-100000 |
| `rating` | float | 1.0-5.0 |
| `latitude` | float | -90~90 |

### 枚举类（固定选项）

| 列名 | 生成器 | 参数 |
|------|--------|------|
| `status` | choice | [0, 1] |
| `gender` | choice | ["male", "female", "other"] |
| `priority` | choice | ["low", "medium", "high"] |
| `role` | choice | ["admin", "user", "guest"] |

In [5]:
rows = preview(str(db_path), table="members", count=3)
for row in rows:
    print(f"name={row['name']}, email={row['email']}, phone={row['phone']}, balance={row['balance']}")
print("\nname/email/phone/balance 全部通过精确匹配自动推断")

name=Monroe Moran, email=terminal2034@yandex.com, phone=+17308896671, balance=221146.24
name=Erik Sampson, email=medium1840@yahoo.com, phone=+17083994550, balance=623384.49
name=Len Burton, email=pmid1935@gmail.com, phone=+17735220223, balance=100396.55

name/email/phone/balance 全部通过精确匹配自动推断


## 6. Level 5: DEFAULT Value

如果列有 DEFAULT 值，sqlseed 默认跳过（使用默认值）。`enrich=True` 时会分析默认值模式。

In [6]:
import sqlite3

conn = sqlite3.connect(str(db_path))
cols = conn.execute("PRAGMA table_info(projects)").fetchall()
default_cols = [c[1] for c in cols if c[4] is not None]
print(f"有 DEFAULT 的列: {default_cols}")
conn.close()

rows = preview(str(db_path), table="projects", count=2)
print(f"preview 输出包含的列: {list(rows[0].keys())}")
skipped = [c for c in default_cols if c not in rows[0]]
print(f"被跳过的 DEFAULT 列: {skipped}")
print("\n有 DEFAULT 的列默认跳过,使用默认值（不生成数据）")

有 DEFAULT 的列: ['budget', 'task_count', 'is_public', 'is_archived']
preview 输出包含的列: ['project_no', 'short_code', 'name', 'org_code', 'created_at', 'description']
被跳过的 DEFAULT 列: ['budget', 'task_count', 'is_public', 'is_archived']

有 DEFAULT 的列默认跳过,使用默认值（不生成数据）


## 7. Level 7: Built-in Pattern Match（26 条正则）

> **注**: Level 3（Custom Exact Match）和 Level 6（Custom Pattern Match）是插件扩展点，通过 `sqlseed_register_column_mappers` Hook 注册自定义规则。详见 [08-plugin-hooks.ipynb](08-plugin-hooks.ipynb) 第 10 节。

当精确匹配失败时，sqlseed 用正则模式匹配列名后缀：

| 模式 | 生成器 | 示例列名 |
|------|--------|----------|
| `.*_id$` | foreign_key_or_integer | `project_id`, `assignee_id` |
| `.*_no$` / `.*_nbr$` | foreign_key_or_integer | `project_no`, `member_no` |
| `.*_at$` | datetime | `created_at`, `due_at` |
| `.*_date$` | date | `birth_date` |
| `^is_.*` / `^has_.*` | boolean | `is_active`, `is_public` |
| `.*_code$` | string (alphanumeric) | `org_code`, `region_code` |
| `.*_name$` | name | `org_name`, `file_name` |
| `.*_count$` / `.*_num$` | integer (0-10000) | `task_count`, `item_num` |
| `.*_amount$` / `.*_price$` | float | `total_amount`, `unit_price` |

In [7]:
# *_no 模式匹配 → foreign_key_or_integer → 由于无 FK 约束,回退为随机字符串 (因为是 VARCHAR)
# org_code 匹配 *_code → 本应是 string(alphanumeric),但自动探测到 FK 约束,被智能升级为 foreign_key 并提取 organizations 已有真实值！  # noqa: E501
# *_at 模式匹配 → datetime
rows = preview(str(db_path), table="projects", count=3)
for row in rows:
    pno = str(row['project_no'])[:20]
    code = str(row['org_code'])[:12]
    print(f"project_no={pno:<20s}  org_code={code:<12s}  created_at={row['created_at']}")
print("\nproject_no → *_no 模式 → foreign_key_or_integer (无 FK,回退为随机字符串)")
print("org_code → 自动侦测到 FK 约束,智能升级为 foreign_key (提取了父表真实值)")
print("created_at → *_at 模式 → datetime")

project_no=NzOES4pWSuBNM         org_code=JmTPSI        created_at=2024-09-20 21:42:26.229719
project_no=OMlZdFi               org_code=oCLrZ3aWZ     created_at=2002-04-19 03:50:52.291483
project_no=2Dn4aMrws1hzXb        org_code=hbVrpoiVgRV   created_at=2006-05-30 07:58:09.416734

project_no → *_no 模式 → foreign_key_or_integer (无 FK,回退为随机字符串)
org_code → 自动侦测到 FK 约束,智能升级为 foreign_key (提取了父表真实值)
created_at → *_at 模式 → datetime


## 8. Level 8: Nullable

可空列（没有 DEFAULT 值）默认跳过。`enrich=True` 时会根据已有数据推断生成策略。

In [8]:
conn = sqlite3.connect(str(db_path))
cols = conn.execute("PRAGMA table_info(members)").fetchall()
nullable_cols = [c[1] for c in cols if c[3] == 0 and c[4] is None]
print(f"Nullable columns (no default): {nullable_cols}")
conn.close()

Nullable columns (no default): ['member_id', 'phone', 'avatar', 'registered_at', 'address']


## 9. Level 9: Type Fallback（22 种 SQL 类型）

当所有命名策略都失败时，sqlseed 根据列的 SQL 类型选择生成器：

| SQL 类型 | 生成器 | 参数 |
|----------|--------|------|
| INTEGER | integer | 0-999999 |
| REAL / FLOAT | float | 0-999999 |
| TEXT | string | 5-50 字符 |
| VARCHAR(n) | string | 1-n 字符 |
| BLOB | bytes | 32 字节 |
| BOOLEAN | boolean | - |
| DATE | date | - |
| DATETIME | datetime | - |

In [9]:
conn = sqlite3.connect(str(db_path))
conn.execute("""CREATE TABLE IF NOT EXISTS demo_fallback (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    info TEXT NOT NULL,
    score_value REAL NOT NULL
)""")
conn.commit()
conn.close()

rows = preview(str(db_path), table="demo_fallback", count=2)
for row in rows:
    info = str(row['info'])[:50]
    print(f"info=\"{info}...\", score_value={row['score_value']:.2f}")
print("\ninfo (TEXT, 无列名匹配) → 类型回退 → 随机字符串 (5-50字符)")
print("score_value (REAL, 无列名匹配) → 类型回退 → 随机浮点数")

# Cleanup
conn = sqlite3.connect(str(db_path))
conn.execute("DROP TABLE IF EXISTS demo_fallback")
conn.commit()
conn.close()

info="7VzI5SB1_qPez2SuTjxv8uu4hf6...", score_value=956096.77
info="OjkddN4RzhMOgLNPzNDXdKvCBfz5uojo...", score_value=548039.88

info (TEXT, 无列名匹配) → 类型回退 → 随机字符串 (5-50字符)
score_value (REAL, 无列名匹配) → 类型回退 → 随机浮点数


## 10. inspect --show-mapping 实战

使用 CLI 的 `inspect --show-mapping` 命令可以查看每列的映射结果：

In [10]:
from click.testing import CliRunner

from sqlseed.cli.main import cli

runner = CliRunner()
result = runner.invoke(cli, ["inspect", str(db_path), "--show-mapping"])
if result.output.strip():
    print(result.output)

                                           Table: organizations (5 rows)                                           
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column       ┃ Type        ┃ Nullable ┃ PK ┃ Auto ┃ Generator   ┃ Params                                        ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ org_code     │ VARCHAR(16) │ ✗        │ ✓  │      │ string      │ {'min_length': 6, 'max_length': 12,           │
│              │             │          │    │      │             │ 'charset': 'alphanumeric'}                    │
│ name         │ VARCHAR(64) │ ✗        │    │      │ name        │ {}                                            │
│ parent_code  │ VARCHAR(16) │ ✓        │    │      │ foreign_key │ {'ref_table': 'organizations', 'ref_column':  │
│              │             │          │    │      │             │ 'org_code', 'strategy': 'random',             │
│              │             │          │    │      │             │ '_ref_values': ['JmTPSI', 'SBvrjn9',          │
│              │             │          │    │      │             │ 'fLBcbfnoGM', 'hbVrpoiVgRV', 'oCLrZ3aWZ']}    │
│ description  │ TEXT        │ ✓        │    │      │ text        │ {'min_length': 100, 'max_length': 500}        │
│ is_active    │ INTEGER     │ ✓        │    │      │ skip        │ {}                                            │
│ member_count │ INTEGER     │ ✓        │    │      │ skip        │ {}                                            │
│ created_at   │ TEXT        │ ✓        │    │      │ datetime    │ {}                                            │
└──────────────┴─────────────┴──────────┴────┴──────┴─────────────┴───────────────────────────────────────────────┘

        Foreign Keys: organizations         
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Column      ┃ Ref Table     ┃ Ref Column ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ parent_code │ organizations │ org_code   │
└─────────────┴───────────────┴────────────┘

                                             Table: members (20 rows)                                              
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column        ┃ Type         ┃ Nullable ┃ PK ┃ Auto ┃ Generator   ┃ Params                                      ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ member_id     │ INTEGER      │ ✗        │ ✓  │ ✓    │ skip        │ {}                                          │
│ member_no     │ VARCHAR(16)  │ ✗        │    │      │ string      │ {'min_length': 4, 'max_length': 20}         │
│ name          │ VARCHAR(64)  │ ✗        │    │      │ name        │ {}                                          │
│ email         │ VARCHAR(128) │ ✗        │    │      │ email       │ {}                                          │
│ phone         │ VARCHAR(20)  │ ✓        │    │      │ phone       │ {}                                          │
│ org_code      │ VARCHAR(16)  │ ✗        │    │      │ foreign_key │ {'ref_table': 'organizations',              │
│               │              │          │    │      │             │ 'ref_column': 'org_code', 'strategy':       │
│               │              │          │    │      │             │ 'random', '_ref_values': ['JmTPSI',         │
│               │              │          │    │      │             │ 'SBvrjn9', 'fLBcbfnoGM', 'hbVrpoiVgRV',     │
│               │              │          │    │      │             │ 'oCLrZ3aWZ']}                               │
│ is_active     │ INTEGER      │ ✓        │    │      │ skip        │ {}                                          │
│ balance       │ REAL         │ ✓        │    │      │ float       │ {'min_value': 0.0, 'max_value': 999999.99,  │
│               │              │          │    │      │             │ 'precision': 2}                             │
│ avatar        │ BLOB         │ ✓        │    │      │ url         │ {}                                          │
│ registered_at │ TEXT         │ ✓        │    │      │ datetime    │ {}                                          │
│ address       │ TEXT         │ ✓        │    │      │ address     │ {}                                          │
└───────────────┴──────────────┴──────────┴────┴──────┴─────────────┴─────────────────────────────────────────────┘

          Foreign Keys: members          
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Column   ┃ Ref Table     ┃ Ref Column ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ org_code │ organizations │ org_code   │
└──────────┴───────────────┴────────────┘

               Table: sqlite_sequence (3 rows)               
┏━━━━━━━━┳━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Column ┃ Type ┃ Nullable ┃ PK ┃ Auto ┃ Generator ┃ Params ┃
┡━━━━━━━━╇━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ name   │      │ ✓        │    │      │ name      │ {}     │
│ seq    │      │ ✓        │    │      │ skip      │ {}     │
└────────┴──────┴──────────┴────┴──────┴───────────┴────────┘

...

                          Table: tags (8 rows)                           
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Column      ┃ Type        ┃ Nullable ┃ PK ┃ Auto ┃ Generator ┃ Params ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ tag_id      │ INTEGER     │ ✗        │ ✓  │ ✓    │ skip      │ {}     │
│ name        │ VARCHAR(32) │ ✗        │    │      │ name      │ {}     │
│ color       │ VARCHAR(7)  │ ✓        │    │      │ skip      │ {}     │
│ usage_count │ INTEGER     │ ✓        │    │      │ skip      │ {}     │
└─────────────┴─────────────┴──────────┴────┴──────┴───────────┴────────┘

                                             Table: task_tags (0 rows)                                             
┏━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column  ┃ Type    ┃ Nullable ┃ PK ┃ Auto ┃ Generator   ┃ Params                                                 ┃
┡━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ task_id │ INTEGER │ ✗        │    │      │ foreign_key │ {'ref_table': 'tasks', 'ref_column': 'task_id',        │
│         │         │          │    │      │             │ 'strategy': 'random', '_ref_values': []}               │
│ tag_id  │ INTEGER │ ✗        │    │      │ foreign_key │ {'ref_table': 'tags', 'ref_column': 'tag_id',          │
│         │         │          │    │      │             │ 'strategy': 'random', '_ref_values': [1, 5, 3, 6, 8,   │
│         │         │          │    │      │             │ 2, 7, 4]}                                              │
└─────────┴─────────┴──────────┴────┴──────┴─────────────┴────────────────────────────────────────────────────────┘

      Foreign Keys: task_tags       
┏━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Column  ┃ Ref Table ┃ Ref Column ┃
┡━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ tag_id  │ tags      │ tag_id     │
│ task_id │ tasks     │ task_id    │
└─────────┴───────────┴────────────┘

                                            Table: attachments (0 rows)                                            
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column        ┃ Type         ┃ Nullable ┃ PK ┃ Auto ┃ Generator   ┃ Params                                      ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ attachment_id │ INTEGER      │ ✗        │ ✓  │ ✓    │ skip        │ {}                                          │
│ task_id       │ INTEGER      │ ✗        │    │      │ foreign_key │ {'ref_table': 'tasks', 'ref_column':        │
│               │              │          │    │      │             │ 'task_id', 'strategy': 'random',            │
│               │              │          │    │      │             │ '_ref_values': []}                          │
│ file_name     │ VARCHAR(128) │ ✗        │    │      │ name        │ {}                                          │
│ file_data     │ BLOB         │ ✓        │    │      │ skip        │ {}                                          │
│ file_size     │ INTEGER      │ ✓        │    │      │ skip        │ {}                                          │
│ uploaded_at   │ TEXT         │ ✓        │    │      │ datetime    │ {}                                          │
└───────────────┴──────────────┴──────────┴────┴──────┴─────────────┴─────────────────────────────────────────────┘

     Foreign Keys: attachments      
┏━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Column  ┃ Ref Table ┃ Ref Column ┃
┡━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ task_id │ tasks     │ task_id    │
└─────────┴───────────┴────────────┘

## 11. 自定义映射规则（Level 3 / Level 6）

通过 `sqlseed_register_column_mappers` 插件 Hook 可以注册自定义规则。这些规则在 Level 3（精确匹配）和 Level 6（模式匹配）中优先于内置规则：

In [11]:
import pluggy

from sqlseed.plugins.hookspecs import SqlseedHookSpec, hookimpl

pm = pluggy.PluginManager("sqlseed")
pm.add_hookspecs(SqlseedHookSpec)

class CustomMapperPlugin:
    @hookimpl
    def sqlseed_register_column_mappers(self, mapper):
        mapper.register_exact_rule("color", "choice", {"choices": ["red", "green", "blue"]})
        mapper.register_pattern_rule(r".*_color$", "choice", {"choices": ["#ff0000", "#00ff00", "#0000ff"]})

pm.register(CustomMapperPlugin())
print("Custom mapper plugin registered")
print("  Exact rule: 'color' → choice([red, green, blue])")
print("  Pattern rule: '*_color' → choice([#ff0000, #00ff00, #0000ff])")

Custom mapper plugin registered
  Exact rule: 'color' → choice([red, green, blue])
  Pattern rule: '*_color' → choice([#ff0000, #00ff00, #0000ff])


## 🎯 enrich 模式：Level 5 和 Level 8 的 __enrich__ 行为

正常情况下，Level 5（有 DEFAULT 值）和 Level 8（可 NULL）的列会被**跳过**。但当 `enrich=True` 时，这些列会进入 `__enrich__` 模式：

- **DEFAULT 列**：如果被 EnrichmentEngine 识别为枚举列，生成有意义的枚举值
- **Nullable 列**：如果被识别为枚举列，生成枚举值；否则按 null_ratio 生成

### EnrichmentEngine 枚举列检测

EnrichmentEngine 使用 19 种列名模式检测枚举列：

| 模式 | 示例列名 |
|---|---|
| `*_status` | order_status, project_status |
| `*_type` | user_type, file_type |
| `is_*` | is_active, is_public |
| `has_*` | has_permission |
| `*_level` | priority_level, access_level |
| `*_category` | product_category |
| `*_flag` | feature_flag |
| `*_mode` | payment_mode |
| ... | 共 19 种模式 |

此外还会通过**基数比**（distinct_count / total_rows < 0.3）和**小整数类型**（INT8/INT16/TINYINT/SMALLINT）来辅助判断。

In [12]:
from sqlseed.core.enrichment import EnrichmentEngine

print("EnrichmentEngine 枚举列名模式 (19种):")
for i, pattern in enumerate(EnrichmentEngine.ENUM_NAME_PATTERNS, 1):
    print(f"  {i:2d}. {pattern}")

print(f"\n小整数类型: {EnrichmentEngine.SMALL_INT_TYPES}")

EnrichmentEngine 枚举列名模式 (19种):
   1. ^[bB]y[A-Za-z]
   2. .*_type$
   3. .*_status$
   4. ^is_.*
   5. ^has_.*
   6. ^can_.*
   7. .*_level$
   8. .*_category$
   9. .*_class$
  10. .*_flag$
  11. .*_kind$
  12. .*_grade$
  13. .*_rank$
  14. .*_tier$
  15. .*_mode$
  16. .*_stage$
  17. .*_phase$
  18. .*_state$
  19. .*_group$

小整数类型: ('INT8', 'INT16', 'TINYINT', 'SMALLINT')


## 🔗 foreign_key_or_integer 智能解析

Level 7 中 `*_id` 和 `*_no` 列会匹配 `foreign_key_or_integer` 生成器。其解析逻辑：

1. **优先查 FK 约束**：如果列有 `FOREIGN KEY` 约束 → 使用 `foreign_key` 生成器
2. **查 SharedPool**：如果 SharedPool 中有同名列的值 → 使用 `foreign_key` 生成器
3. **按列类型回退**：INTEGER 类型 → `integer` 生成器，其他 → `string` 生成器

In [13]:
with sqlseed.connect(str(db_path)) as orch:
    col_info = orch.get_column_info("members")
    fk_info = orch.get_foreign_keys("members")

    fk_columns = {fk.column for fk in fk_info}
    print("members 表列映射分析:")
    for col in col_info:
        if col.name.endswith(("_id", "_no")):
            is_fk = col.name in fk_columns
            gen_type = "foreign_key" if is_fk else "integer/string"
            print(f"  {col.name}: {gen_type} (FK={is_fk})")

members 表列映射分析:
  member_id: integer/string (FK=False)
  member_no: integer/string (FK=False)


## 12. 总结

| 级别 | 策略 | 规则数 | 优先级 |
|:----:|------|:------:|:------:|
| 1 | Autoincrement PK | - | 最高 |
| 2 | User Config | 无限 | 很高 |
| 3 | Custom Exact Match | 插件注册 | 高 |
| 4 | Built-in Exact Match | 74 | 高 |
| 5 | DEFAULT Value | - | 中高 |
| 6 | Custom Pattern Match | 插件注册 | 中 |
| 7 | Built-in Pattern Match | 26 | 中 |
| 8 | Nullable | - | 中低 |
| 9 | Type Fallback | 22 | 最低 |

**关键洞察**：
- 命名比类型更重要——`email VARCHAR(128)` 会生成邮箱而非随机字符串
- 用户配置可以覆盖一切自动推断
- 插件可以通过 Level 3/6 注入自定义规则，优先于内置规则
- `inspect --show-mapping` 是调试映射问题的第一工具

**下一步**: [03-generators.ipynb](03-generators.ipynb) — 了解 31 种生成器和 Provider 体系

In [14]:
# ✅ 验证: 确保数据已成功生成并写入
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # 基本行数验证
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
